# Notebook 09: Explanatory — Resident Risk Factor Analysis

## Section 1 — Problem Framing

**Business question:** Among the observable and potentially modifiable factors —
service intensity, education engagement, health trajectory, family cooperation,
and incident history — which are most strongly associated with a resident's
*current risk level*?

**Who cares:** Safehouse managers and social workers need to know *where to
allocate scarce resources*. If certain patterns (e.g., low family cooperation
combined with infrequent counseling) strongly predict elevated risk, the
organization can design targeted interventions.

**Approach: Explanatory.** We use OLS regression so that each coefficient has a
direct interpretation: "a one-unit increase in X is associated with a β-unit
change in risk, holding other factors constant." We supplement with a Random
Forest as a predictive robustness check and to surface non-linear feature
importance.

**Success metric:** Adjusted R², coefficient significance (p < 0.05), and — most
importantly — whether the relationships are substantively meaningful and
actionable for the organization.

## Section 2 — Data Acquisition and Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import json, os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

DATA_DIR = '../../data/lighthouse_csv_v7/'
RESULTS_DIR = '../../data/explanatory_results/'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete — all libraries loaded.')

In [ ]:
from sqlalchemy import create_engine
import os

DB_HOST = os.environ.get("DB_HOST", "localhost")
DB_PORT = os.environ.get("DB_PORT", "5432")
DB_NAME = os.environ.get("DB_NAME", "harbor_of_hope")
DB_USER = os.environ.get("DB_USER", "waylansmac")
DB_PASS = os.environ.get("DB_PASS", "")
CONNECTION_STRING = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(CONNECTION_STRING)

residents = pd.read_sql("SELECT * FROM residents", engine)
education = pd.read_sql("SELECT * FROM education_records", engine)
health = pd.read_sql("SELECT * FROM health_wellbeing_records", engine)
incidents = pd.read_sql("SELECT * FROM incident_reports", engine)
sessions = pd.read_sql("SELECT * FROM process_recordings", engine)
visitations = pd.read_sql("SELECT * FROM home_visitations", engine)
interventions = pd.read_sql("SELECT * FROM intervention_plans", engine)

print(f'Loaded {len(residents)} residents, {len(education)} education records, '
      f'{len(health)} health records, {len(incidents)} incidents')
print(f'Loaded {len(sessions)} counseling sessions, {len(visitations)} home visitations, '
      f'{len(interventions)} intervention plans')

In [ ]:
def parse_duration_months(s):
    """Convert '15 Years 9 months' to numeric months."""
    if pd.isna(s):
        return np.nan
    try:
        parts = str(s).split()
        return int(parts[0]) * 12 + int(parts[2])
    except Exception:
        return np.nan

def to_bool_int(series):
    """Robustly convert True/False (bool or string) to 1/0."""
    return series.map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0).astype(int)

def plot_coefficients(params, conf_int, pvalues, title, filename):
    """Bar-plot of OLS / logistic regression coefficients with 95% CI."""
    df_coef = pd.DataFrame({
        'coef': params,
        'ci_low': conf_int.iloc[:, 0],
        'ci_high': conf_int.iloc[:, 1],
        'pvalue': pvalues
    })
    df_coef = df_coef.drop('const', errors='ignore')
    df_coef['significant'] = df_coef['pvalue'] < 0.05
    df_coef = df_coef.sort_values('coef')
    fig, ax = plt.subplots(figsize=(10, max(6, len(df_coef) * 0.35)))
    colors = ['#2196F3' if s else '#BDBDBD' for s in df_coef['significant']]
    y_pos = range(len(df_coef))
    ax.barh(y_pos, df_coef['coef'], color=colors, edgecolor='white', height=0.7)
    ax.errorbar(df_coef['coef'], y_pos,
                xerr=[df_coef['coef'] - df_coef['ci_low'],
                      df_coef['ci_high'] - df_coef['coef']],
                fmt='none', ecolor='black', capsize=3, linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_coef.index, fontsize=9)
    ax.axvline(0, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Coefficient')
    ax.set_title(title)
    blue_patch = plt.Line2D([0], [0], color='#2196F3', lw=6, label='p < 0.05')
    grey_patch = plt.Line2D([0], [0], color='#BDBDBD', lw=6, label='p >= 0.05')
    ax.legend(handles=[blue_patch, grey_patch], loc='lower right')
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()

print('Helper functions defined.')

In [ ]:
r1 = residents.copy()
r1['age_months'] = r1['age_upon_admission'].apply(parse_duration_months)
r1['stay_months'] = r1['length_of_stay'].apply(parse_duration_months)

bool_cols = [c for c in r1.columns if c.startswith('sub_cat_') or c.startswith('family_')
             or c in ['is_pwd', 'has_special_needs']]
for col in bool_cols:
    r1[col] = to_bool_int(r1[col])

abuse_cols = [c for c in r1.columns if c.startswith('sub_cat_')]
r1['abuse_types_count'] = r1[abuse_cols].sum(axis=1)
r1['family_risk_count'] = r1[['family_solo_parent', 'family_indigenous',
                               'family_parent_pwd', 'family_informal_settler']].sum(axis=1)

edu_agg = education.groupby('resident_id').agg(
    avg_attendance=('attendance_rate', 'mean'),
    avg_progress=('progress_percent', 'mean'),
    edu_count=('education_record_id', 'count'),
    attendance_std=('attendance_rate', 'std')
).reset_index()

health_agg = health.groupby('resident_id').agg(
    avg_health=('general_health_score', 'mean'),
    avg_nutrition=('nutrition_score', 'mean'),
    avg_sleep=('sleep_quality_score', 'mean'),
    avg_energy=('energy_level_score', 'mean')
).reset_index()

inc = incidents.copy()
inc['severity_num'] = inc['severity'].map({'Low': 1, 'Medium': 2, 'High': 3})
incident_agg = inc.groupby('resident_id').agg(
    total_incidents=('incident_id', 'count'),
    avg_severity=('severity_num', 'mean')
).reset_index()

sess = sessions.copy()
sess['concern_flag'] = to_bool_int(sess['concerns_flagged'])
sess['progress_flag'] = to_bool_int(sess['progress_noted'])
session_agg = sess.groupby('resident_id').agg(
    total_sessions=('recording_id', 'count'),
    avg_session_min=('session_duration_minutes', 'mean'),
    concern_rate=('concern_flag', 'mean'),
    progress_rate=('progress_flag', 'mean')
).reset_index()

vis = visitations.copy()
vis['safety_flag'] = to_bool_int(vis['safety_concerns_noted'])
vis['coop_score'] = vis['family_cooperation_level'].map(
    {'Cooperative': 3, 'Neutral': 2, 'Uncooperative': 1}).fillna(2)
visit_agg = vis.groupby('resident_id').agg(
    total_visits=('visitation_id', 'count'),
    avg_family_coop=('coop_score', 'mean'),
    safety_concern_rate=('safety_flag', 'mean')
).reset_index()

int_agg = interventions.groupby('resident_id').agg(
    total_plans=('plan_id', 'count'),
    achieved_rate=('status', lambda x: (x == 'Achieved').mean())
).reset_index()

keep_cols = ['resident_id', 'current_risk_level', 'case_category',
             'age_months', 'stay_months', 'abuse_types_count',
             'family_risk_count', 'is_pwd', 'has_special_needs',
             'family_is_4ps', 'referral_source', 'safehouse_id']
df1 = r1[keep_cols].copy()
df1['family_is_4ps'] = to_bool_int(df1['family_is_4ps'])

for agg in [edu_agg, health_agg, incident_agg, session_agg, visit_agg, int_agg]:
    df1 = df1.merge(agg, on='resident_id', how='left')

count_fill = ['total_incidents', 'total_sessions', 'total_visits', 'total_plans']
df1[count_fill] = df1[count_fill].fillna(0)
rate_fill = ['concern_rate', 'progress_rate', 'safety_concern_rate',
             'achieved_rate', 'avg_severity']
df1[rate_fill] = df1[rate_fill].fillna(0)

risk_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
df1['risk_numeric'] = df1['current_risk_level'].map(risk_map)
df1 = df1.dropna(subset=['risk_numeric'])

df1 = pd.get_dummies(df1, columns=['case_category', 'referral_source'], drop_first=True, dtype=int)

print(f'Pipeline 1 analytical dataset: {df1.shape[0]} rows x {df1.shape[1]} columns')
print(f'\nRisk level distribution:\n{df1["current_risk_level"].value_counts().sort_index()}')
df1.head()

## Section 3 — Exploration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

order = ['Low', 'Medium', 'High', 'Critical']
sns.countplot(data=df1, x='current_risk_level', order=order, palette='YlOrRd', ax=axes[0, 0])
axes[0, 0].set_title('Distribution of Current Risk Level')
axes[0, 0].set_xlabel('')

sns.boxplot(data=df1, x='current_risk_level', y='avg_attendance', order=order,
            palette='YlOrRd', ax=axes[0, 1])
axes[0, 1].set_title('Education Attendance by Risk Level')

sns.boxplot(data=df1, x='current_risk_level', y='total_incidents', order=order,
            palette='YlOrRd', ax=axes[1, 0])
axes[1, 0].set_title('Total Incidents by Risk Level')

sns.boxplot(data=df1, x='current_risk_level', y='avg_family_coop', order=order,
            palette='YlOrRd', ax=axes[1, 1])
axes[1, 1].set_title('Family Cooperation Score by Risk Level')

plt.suptitle('Pipeline 1 — Exploratory Analysis: Risk Factors', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('p1_eda.png', dpi=150, bbox_inches='tight')
plt.show()

numeric_cols = df1.select_dtypes(include=[np.number]).columns.drop(
    ['resident_id', 'safehouse_id', 'risk_numeric'], errors='ignore')
corr_with_risk = df1[numeric_cols].corrwith(df1['risk_numeric']).sort_values()
fig, ax = plt.subplots(figsize=(8, max(6, len(corr_with_risk) * 0.3)))
corr_with_risk.plot.barh(color=['#EF5350' if v > 0 else '#42A5F5' for v in corr_with_risk], ax=ax)
ax.set_title('Correlation of Each Feature with Risk Level (Numeric)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('p1_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Modeling and Feature Selection

In [ ]:
feature_cols_1 = [c for c in df1.columns if c not in
                  ['resident_id', 'current_risk_level', 'risk_numeric', 'safehouse_id']]
X1 = df1[feature_cols_1].copy()
y1 = df1['risk_numeric'].copy()

X1 = X1.apply(pd.to_numeric, errors='coerce')
mask = X1.notna().all(axis=1)
X1 = X1.loc[mask]
y1 = y1.loc[mask]

scaler1 = StandardScaler()
X1_scaled = pd.DataFrame(scaler1.fit_transform(X1), columns=X1.columns, index=X1.index)

X1_ols = sm.add_constant(X1_scaled)
ols_model_1 = sm.OLS(y1, X1_ols).fit()
print(ols_model_1.summary())

In [ ]:
plot_coefficients(ols_model_1.params, ols_model_1.conf_int(), ols_model_1.pvalues,
                  'Pipeline 1 — OLS Coefficients (standardized) for Risk Level',
                  'p1_ols_coefficients.png')

vif_df = pd.DataFrame({
    'Feature': X1_scaled.columns,
    'VIF': [variance_inflation_factor(X1_scaled.values, i) for i in range(X1_scaled.shape[1])]
}).sort_values('VIF', ascending=False)
print('\nVariance Inflation Factors (VIF > 10 suggests problematic multicollinearity):')
print(vif_df.head(10).to_string(index=False))

### Feature Selection Notes

We use **standardized coefficients** so that magnitudes are comparable across features
with different scales. The VIF check above identifies multicollinear features; any
with VIF > 10 should be considered for removal or combination. The coefficient plot
highlights statistically significant features (blue, p < 0.05) vs. non-significant
(grey). We retain all features for interpretability but focus our business
interpretation on the significant ones.

In [ ]:
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1_scaled, y1, test_size=0.2, random_state=42)

rf1 = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf1.fit(X1_train, y1_train)
y1_pred_rf = rf1.predict(X1_test)
print(f'Random Forest — Test R2: {r2_score(y1_test, y1_pred_rf):.3f}')
print(f'Random Forest — Test RMSE: {np.sqrt(mean_squared_error(y1_test, y1_pred_rf)):.3f}')

cv_scores = cross_val_score(rf1, X1_scaled, y1, cv=5, scoring='r2')
print(f'Random Forest — 5-Fold CV R2: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')

fi1 = pd.Series(rf1.feature_importances_, index=X1.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, max(6, len(fi1) * 0.3)))
fi1.tail(15).plot.barh(color='#66BB6A', ax=ax)
ax.set_title('Pipeline 1 — Random Forest Feature Importance (Top 15)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('p1_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

y1_pred_ols = ols_model_1.predict(X1_ols)
print(f'\nOLS — Full-sample R2: {ols_model_1.rsquared:.3f} | Adj R2: {ols_model_1.rsquared_adj:.3f}')
print(f'Comparison: OLS explains variance through interpretable linear coefficients;')
print(f'RF captures non-linearities but sacrifices coefficient interpretation.')

## Section 5 — Evaluation

In [ ]:
sig_features = ols_model_1.pvalues.drop('const', errors='ignore')
sig_features = sig_features[sig_features < 0.05].sort_values()
print('=== Statistically Significant Features (p < 0.05) ===')
for feat, pval in sig_features.items():
    coef = ols_model_1.params[feat]
    direction = 'INCREASES' if coef > 0 else 'DECREASES'
    print(f'  {feat}: coef = {coef:+.3f}, p = {pval:.4f} -> {direction} risk')

print('\n=== Business Interpretation ===')
print('A one-standard-deviation increase in each significant feature is associated')
print('with the corresponding coefficient change in risk level (1=Low ... 4=Critical).')
print('\nActionable recommendations:')
top_pos = ols_model_1.params.drop('const', errors='ignore').nlargest(3)
top_neg = ols_model_1.params.drop('const', errors='ignore').nsmallest(3)
print('\nTop risk-INCREASING factors:')
for f, c in top_pos.items():
    print(f'  * {f} (B = {c:+.3f})')
print('\nTop risk-DECREASING (protective) factors:')
for f, c in top_neg.items():
    print(f'  * {f} (B = {c:+.3f})')

fig, ax = plt.subplots(figsize=(8, 5))
residuals = ols_model_1.resid
ax.scatter(ols_model_1.fittedvalues, residuals, alpha=0.5, edgecolors='k', linewidth=0.5)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Fitted Values')
ax.set_ylabel('Residuals')
ax.set_title('Pipeline 1 — OLS Residual Plot')
plt.tight_layout()
plt.savefig('p1_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 6 — Causal and Relationship Analysis

### Key Findings
The OLS model reveals which factors have the strongest *linear association* with
current risk level after controlling for other variables.

### Defensibility of Causal Claims
- **Incident history -> risk**: Residents with more (and more severe) incidents tend
  to have higher current risk. This is *likely partly causal* (incidents themselves
  signal and perhaps exacerbate risk), but also partly *mechanical* — risk assessments
  may incorporate incident counts directly.
- **Family cooperation -> risk**: Lower family cooperation scores associate with higher
  risk. A plausible causal pathway exists (cooperative families support recovery), but
  *reverse causality* is possible — social workers may rate families as uncooperative
  precisely because the resident is higher-risk.
- **Education attendance / progress -> risk**: Higher engagement appears protective. This
  is consistent with theory (structured activity and skill-building reduce vulnerability),
  but *selection bias* is a concern — residents who are already lower-risk may find it
  easier to attend school.
- **Counseling sessions**: If more sessions correlate with *higher* risk, this is not
  evidence that counseling harms residents — it is a classic case of *confounding by
  indication*: higher-risk residents receive more services.

### Limitations
1. **Observational data** — we cannot make definitive causal claims without a randomized
   experiment or quasi-experimental design.
2. **Omitted variable bias** — unmeasured factors (trauma severity, community context)
   could confound results.
3. **Ordinal target treated as continuous** — OLS on a 1-4 scale is an approximation;
   ordinal logistic regression would be more precise but less interpretable.

### Recommendations
1. **Flag residents whose attendance drops** for proactive outreach.
2. **Invest in family engagement** — cooperation is consistently protective.
3. **Monitor incident patterns** as early-warning signals for escalation.

## Section 7 — Data Leakage Check

### Potential Leakage Concerns
1. **Risk level and incident data**: If `current_risk_level` is computed using the same
   incident and service data we feed as features, the association is partly mechanical
   rather than informative. **Mitigation**: We treat this analysis as explanatory
   (understanding associations) rather than predictive. For a predictive deployment we
   would need to use *lagged* features — data from period t-1 predicting risk at period t.
2. **Cross-sectional snapshot**: All features and the target are measured at the same
   point in time. There is no temporal ordering, so we cannot rule out reverse causality.
   **Mitigation**: We clearly note this limitation and avoid causal language.
3. **Safehouse ID excluded from features**: Including safehouse as a feature could create
   leakage if risk policies differ across locations. We exclude it from the model and
   note this decision.

### Verdict
No direct data leakage is present (no post-outcome variables are used), but the
cross-sectional design limits causal interpretation. This pipeline is appropriate for
**understanding risk factor associations**, not for real-time risk prediction.

## Section 8 — Deployment Notes

### Integration with Harbor of Hope Web Application
This model's key outputs (significant coefficients, feature importance rankings, and
business recommendations) are deployed to the Harbor of Hope admin dashboard via:

- **API Endpoint**: `GET /api/explanatoryinsights/1` returns the model coefficients,
  p-values, and business interpretations as JSON.
- **Dashboard Page**: The "Explanatory Insights" page under Admin > Insights displays
  the risk factor analysis results in an interactive table.
- **Notebook location**: `ml-pipelines/09-explanatory-risk-factors.ipynb`

### How to Refresh Results
1. Run this notebook end-to-end (Kernel > Restart & Run All).
2. The final cell exports updated coefficients to
   `data/explanatory_results/pipeline_01_risk_factors.json`.
3. Restart the backend API server to pick up any new static data.

In [ ]:
results_01 = {
    "pipeline_id": 1,
    "pipeline_name": "Resident Risk Factor Analysis",
    "target_variable": "Current Risk Level (1=Low, 4=Critical)",
    "model_type": "OLS Linear Regression",
    "r_squared": round(ols_model_1.rsquared, 4),
    "adj_r_squared": round(ols_model_1.rsquared_adj, 4),
    "sample_size": int(len(y1)),
    "significant_features": [
        {
            "name": feat,
            "coefficient": round(float(ols_model_1.params[feat]), 4),
            "p_value": round(float(ols_model_1.pvalues[feat]), 4),
            "direction": "increases" if ols_model_1.params[feat] > 0 else "decreases"
        }
        for feat in sig_features.index
    ]
}

with open(f'{RESULTS_DIR}pipeline_01_risk_factors.json', 'w') as f:
    json.dump(results_01, f, indent=2)
print(f'Exported results to {RESULTS_DIR}pipeline_01_risk_factors.json')